In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import re
import time, random
from bs4 import BeautifulSoup
import pandas as pd
import json

In [56]:
#Noticias:

GOOGLE_PESQUISAS = [
   "https://www.google.com/search?q=endividamento+do+brasileiro",
    "https://www.google.com/search?q=voos+internacionais+brasil+noticias"]
    
#EMBRATUR_VOOS_INTERNACIONAIS = "https://embratur.com.br/tag/voos-internacionais/"

In [57]:
def EXTRAIR_CONTEUDO_NOTICIA(link_noticia):
    html = BeautifulSoup(pagina_html, "html.parser")
    texto_limpo = html.get_text(separator=' ', strip=True)
    conteudo_noticias.append({
                "id_pesquisa": link_noticia,
                "html": texto_limpo
            })
    

def EXTRAIR_DADOS_NOTICIAS(pagina_html):
    regex_data = re.compile(r"\d{1,2} de [a-zç\.]+ de \d{4}", re.IGNORECASE)
    soup = BeautifulSoup(pagina_html, "html.parser")
    info_noticias = soup.select("div.MjjYud") 
    
    for info in info_noticias:

        h3 = info.select_one("h3")
        titulo = h3.get_text(strip=True) if h3 else None
        
        a = h3.find_parent("a") if h3 else info.select_one("a[href]")
        link_noticia = a.get("href") if a else None
        #CHAMAR METODO ABRIR NOTICIA E PEGAR TEXTO PRA SALVAR EM JSON
        EXTRAIR_CONTEUDO_NOTICIA(link_noticia) if link_noticia else None
    
        desc = info.select_one(".VwiC3b")
        descricao = desc.get_text(" ", strip=True) if desc else None
    
        data = info.select_one(".YrbPuc")
        data_text = data.get_text(strip=True) if data else None
        data_publicacao = data_text if data_text and regex_data.search(data_text) else None
        
        if titulo and link_noticia:
            dados_noticias.append({
                "titulo": titulo,
                "link": link_noticia,
                "descricao": descricao,
                "data_publicacao": data_publicacao
            })

In [58]:


headless=False #Visibilidade do navegador
timeout=20
options = Options()

#Remove tags na parte superior do site onde diz que é uma automaçao e outras configs que ajudam a mascarar
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--disable-notifications")  # bloqueia pop-ups de notificação
options.add_argument("--disable-popup-blocking")  # desativa bloqueador de pop-ups

#inicia configs do chrome
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

#Adiciona um tempo de espera para os elementos do site
wait = WebDriverWait(driver, timeout=5)

dados_noticias = []
conteudo_noticias = []

for pesquisa in GOOGLE_PESQUISAS:
    driver.get(pesquisa)
    
    #carrega todas as divs que possuem essa tag
    #Informacoes da pag 1
    pagina_principal = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.MjjYud")))
    pagina_html = driver.page_source
    EXTRAIR_DADOS_NOTICIAS(pagina_html)
    
    paginacao = wait.until(EC.presence_of_element_located((By.TAG_NAME, "tbody")))
    links = paginacao.find_elements(By.TAG_NAME, "a")
    lista_links_paginas = [link.get_attribute("href") for link in links]
    
    for pagina in lista_links_paginas:
        print(pagina) 
        driver.get(pagina)
        pagina_html = driver.page_source
        EXTRAIR_DADOS_NOTICIAS(pagina_html)

        driver.quit()


https://www.google.com/search?q=endividamento+do+brasileiro&sca_esv=50bc830c1331a679&ei=A0T-aID0Jefe5OUPkJTV6Qk&start=10&sa=N&sstk=Af77f_frahQZf5zWEw8pIVK1g79_BY5IPtlIR0fwg7C0w8uUlvwOGiZPeLGXfHlbIm1L_QgkVfB92mhPU_uXJhyh2EgXRgOvNAPXMQ&ved=2ahUKEwjAgcHKnMKQAxVnL7kGHRBKNZ0Q8tMDegQIDBAE
https://www.google.com/search?q=endividamento+do+brasileiro&sca_esv=50bc830c1331a679&ei=A0T-aID0Jefe5OUPkJTV6Qk&start=20&sa=N&sstk=Af77f_frahQZf5zWEw8pIVK1g79_BY5IPtlIR0fwg7C0w8uUlvwOGiZPeLGXfHlbIm1L_QgkVfB92mhPU_uXJhyh2EgXRgOvNAPXMQ&ved=2ahUKEwjAgcHKnMKQAxVnL7kGHRBKNZ0Q8tMDegQIDBAG
https://www.google.com/search?q=endividamento+do+brasileiro&sca_esv=50bc830c1331a679&ei=A0T-aID0Jefe5OUPkJTV6Qk&start=30&sa=N&sstk=Af77f_frahQZf5zWEw8pIVK1g79_BY5IPtlIR0fwg7C0w8uUlvwOGiZPeLGXfHlbIm1L_QgkVfB92mhPU_uXJhyh2EgXRgOvNAPXMQ&ved=2ahUKEwjAgcHKnMKQAxVnL7kGHRBKNZ0Q8tMDegQIDBAI
https://www.google.com/search?q=endividamento+do+brasileiro&sca_esv=50bc830c1331a679&ei=A0T-aID0Jefe5OUPkJTV6Qk&start=40&sa=N&sstk=Af77f_frahQZf5zWE

<H2> BASE DE DADOS SOBRE A PESQUISA EXTRAIDA PARA INFORMACOES RESUMIDAS DA SEGUNDA BASE - CHAVE - link

In [59]:
df_noticias = pd.DataFrame(dados_noticias)

In [60]:
df_noticias

,titulo,link,descricao,data_publicacao
0,Inadimplência atinge em setembro maior patamar da série ...,https://www.cnnbrasil.com.br/economia/macroeconomia/inadimplencia-atinge-em-setembro-maior-patamar-da-serie-historica-diz-cnc/,"8 de out. de 2025 — A proporção de famílias com contas em atraso subiu a 30,5% em setembro, maior patamar da série histórica iniciada em 2010, apontou a Peic ( ...",8 de out. de 2025—
1,Mapa da Inadimplência e Negociação de Dívidas no Brasil,https://www.serasa.com.br/limpa-nome-online/blog/mapa-da-inadimplencia-e-renogociacao-de-dividas-no-brasil/,"A inadimplência no Brasil cresceu pelo segundo mês consecutivo . O aumento é de 1,19% em relação ao mês anterior, que corresponde a um acréscimo de 855 mil no ...",None
2,"CNC: endividamento das famílias sobe a 78,4% em junho",https://www.cnnbrasil.com.br/economia/macroeconomia/endividamento-sobe-a-784-das-familias-em-junho-e-inadimplencia-estabiliza-em-295-diz-cnc/,"3 de jul. de 2025 — A proporção de famílias com contas a vencer cresceu de 78,2% em maio para 78,4% em jun ho, o quinto mês consecutivo de altas, apontou a Pesquisa ...",3 de jul. de 2025—
3,Lula deixará contas públicas em crise ao fim do mandato - Gazeta do Povo,"https://www.gazetadopovo.com.br/economia/crise-fiscal-heranca-lula-2027/#:~:text=Desde%20que%20Lula%20assumiu%2C%20em,75%2C9%25%20do%20PIB.",None,None
4,Inadimplência cresce e atinge maior patamar em quase dois ...,https://g1.globo.com/economia/noticia/2025/08/07/inadimplencia-cresce-e-atinge-maior-patamar-em-quase-dois-anos-aponta-cnc.ghtml,"7 de ago. de 2025 — Proporção de famílias com dívidas e contas em atraso chegou a 30,2% da população brasileira — nível mais alto desde setembro de 2023.",7 de ago. de 2025—
...,...,...,...,...
211,Senado se antecipa à Câmara e aprova bagagem de mão ...,https://exame.com/brasil/senado-se-antecipa-a-camara-e-aprova-bagagem-de-mao-gratuita-em-voos/,"há 4 dias — — Nós aprovamos um projeto que impedia a cobrança em bagagens despachadas, e ele foi vetado sob o argumento de que isso baratearia as passagens ...",None
212,Brasileiros priorizam viagens para América do Sul em ...,https://www.cnnbrasil.com.br/economia/brasileiros-priorizam-viagens-para-america-do-sul-em-2025-diz-governo/,20 de set. de 2025 — Siga a CNN Brasil no Google e receba as principais notícias do Brasil e do Mundo ... CNN Brasil MoneyANAC Voos internacionais . Mais Lidas ...,20 de set. de 2025—
213,Brasil tem resultado histórico em decolagens ...,https://www.gov.br/portos-e-aeroportos/pt-br/assuntos/noticias/2025/07/brasil-tem-resultado-historico-em-decolagens-internacionais-nos-primeiros-meses-de-2025,"3 de jul. de 2025 — De janeiro a maio, crescimento de voos para o exterior foi 15,6% maior do que no mesmo período do ano passado.",3 de jul. de 2025—
214,Brasil vai ganhar a partir de agosto nove voos internacionais,https://turismo.ig.com.br/colunas/celso-martins/2025-07-28/brasil-vai-ganhar-a-partir-de-agosto-nove-voos-internacionais.html,28 de jul. de 2025 — A região Sul do Brasil é a que terá o maior número de rotas novas para destinos internacionais . Florianópolis terá voos sem escalas da Latam a ...,28 de jul. de 2025—


<H2> BASE HTML EXTRAIDA PARA ARMAZENAMENTO DE TEXTO E LEITURA DE IA PARA RESUMO E ENTENDIMENTO SOBRE A PESQUISA

In [61]:
df_conteudo_noticias = pd.DataFrame(conteudo_noticias)

In [62]:
df_conteudo_noticias

id_pesquisa  \
0                                    https://www.cnnbrasil.com.br/economia/macroeconomia/inadimplencia-atinge-em-setembro-maior-patamar-da-serie-historica-diz-cnc/   
1                                                       https://www.serasa.com.br/limpa-nome-online/blog/mapa-da-inadimplencia-e-renogociacao-de-dividas-no-brasil/   
2                     https://www.cnnbrasil.com.br/economia/macroeconomia/endividamento-sobe-a-784-das-familias-em-junho-e-inadimplencia-estabiliza-em-295-diz-cnc/   
3                       https://www.gazetadopovo.com.br/economia/crise-fiscal-heranca-lula-2027/#:~:text=Desde%20que%20Lula%20assumiu%2C%20em,75%2C9%25%20do%20PIB.   
4                                  https://g1.globo.com/economia/noticia/2025/08/07/inadimplencia-cresce-e-atinge-maior-patamar-em-quase-dois-anos-aponta-cnc.ghtml   
..                                                                                                                                                              ...   
227                                                                  https://exame.com/brasil/senado-se-antecipa-a-camara-e-aprova-bagagem-de-mao-gratuita-em-voos/   
228                                                    https://www.cnnbrasil.com.br/economia/brasileiros-priorizam-viagens-para-america-do-sul-em-2025-diz-governo/   
229  https://www.gov.br/portos-e-aeroportos/pt-br/assuntos/noticias/2025/07/brasil-tem-resultado-historico-em-decolagens-internacionais-nos-primeiros-meses-de-2025   
230                                   https://turismo.ig.com.br/colunas/celso-martins/2025-07-28/brasil-vai-ganhar-a-partir-de-agosto-nove-voos-internacionais.html   
231                                    https://maringapost.com.br/poder/2025/10/25/aeroporto-de-maringa-planeja-internacionalizacao-para-atender-aviacao-executiva/   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

<H2> Salve em Json - Simulando um ambiente em que há um DataLake e arquitetura de medalhão, essa seria a BRONZE

In [63]:
df_noticias.to_json("noticias.json", orient="records", indent=4, force_ascii=False)

In [64]:
df_conteudo_noticias.to_json("noticias_html.json", orient="records", indent=4, force_ascii=False, default_handler=str)